In [9]:
from pathlib import Path

DOWNLOAD_DIR = Path(
    "/kaggle/working/mooccubex_complete/relations"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

COMPLETE_USER_VIDEO = (
    DOWNLOAD_DIR / "user-video.json"
)

print("Download location:", COMPLETE_USER_VIDEO)

Download location: /kaggle/working/mooccubex_complete/relations/user-video.json


In [10]:
!curl -L --fail \
    --retry 20 \
    --retry-all-errors \
    --connect-timeout 30 \
    --speed-time 120 \
    --speed-limit 1024 \
    -C - \
    "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/user-video.json" \
    -o "/kaggle/working/mooccubex_complete/relations/user-video.json"

** Resuming transfer from byte position 3179221944
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   203    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (22) The requested URL returned error: 416
  0   203    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (22) The requested URL returned error: 416
  0   203    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (22) The requested URL returned error: 416
  0   203    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (22) The requested URL returned error: 416
  0   203    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (22) The requested URL returned error: 416
^C


In [11]:
%%bash
set -e

DEST="/kaggle/working/mooccubex_complete/relations"
FILE="$DEST/user-video.json"
URL="https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/user-video.json"

mkdir -p "$DEST"

echo "Free disk space:"
df -h /kaggle/working

curl -L --fail \
  --retry 20 \
  --retry-all-errors \
  --retry-delay 5 \
  --connect-timeout 30 \
  --speed-time 120 \
  --speed-limit 1024 \
  -C - \
  "$URL" \
  -o "$FILE"

echo "Downloaded file:"
ls -lh "$FILE"

ACTUAL_SIZE=$(stat -c%s "$FILE")
echo "Size in bytes: $ACTUAL_SIZE"

if [ "$ACTUAL_SIZE" -lt 3000000000 ]; then
    echo "WARNING: The file appears incomplete. Run this cell again to resume."
    exit 1
fi

echo "Download completed successfully."

Process is interrupted.


In [12]:
from pathlib import Path
import json

USER_VIDEO = Path(
    "/kaggle/working/mooccubex_complete/relations/user-video.json"
)

print("Exists:", USER_VIDEO.exists())
print("Size:", round(USER_VIDEO.stat().st_size / 1024**3, 3), "GiB")

with USER_VIDEO.open("r", encoding="utf-8") as f:
    first_character = f.read(1)
    f.seek(0)

    if first_character == "[":
        first_record = next(iter(json.load(f)))
    else:
        first_record = json.loads(f.readline())

print("First record keys:", first_record.keys())

Exists: True
Size: 2.961 GiB
First record keys: dict_keys(['seq', 'user_id'])


In [13]:
from pathlib import Path
import duckdb
import os
import gc

# Close an old DuckDB connection left by the previous run.
if "con" in globals():
    try:
        con.close()
        print("Closed previous DuckDB connection.")
    except Exception as error:
        print("Previous connection was already closed:", error)

gc.collect()

DB = Path("/kaggle/working/mooccubex_preprocessing.duckdb")

if DB.exists():
    DB.unlink()
    print("Deleted stale DuckDB:", DB)

Path("/kaggle/working/duckdb_temp").mkdir(
    parents=True,
    exist_ok=True
)

con = duckdb.connect(str(DB))

con.execute(
    f"PRAGMA threads={max(1, min(4, os.cpu_count() or 2))}"
)
con.execute("PRAGMA memory_limit='12GB'")
con.execute(
    "PRAGMA temp_directory='/kaggle/working/duckdb_temp'"
)

stale_tables = [
    "interactions",
    "core",
    "core_next",
    "ranked",
    "train_base",
    "valid_base",
    "test_base",
    "eval_users",
    "eval_users_next",
    "train",
    "valid",
    "test",
]

for table in stale_tables:
    con.execute(f"DROP TABLE IF EXISTS {table}")

con.execute(
    f"""
    CREATE TABLE interactions AS
    SELECT *
    FROM read_parquet(
        '{RAW_INTERACTIONS.as_posix()}'
    )
    WHERE positive = 1
    """
)

positive_rows = con.execute(
    "SELECT COUNT(*) FROM interactions"
).fetchone()[0]

if positive_rows == 0:
    raise RuntimeError(
        "No positive interactions were found."
    )

print("DuckDB initialized successfully.")
print(f"Positive interactions: {positive_rows:,}")

Closed previous DuckDB connection.
Deleted stale DuckDB: /kaggle/working/mooccubex_preprocessing.duckdb


KeyboardInterrupt: 

In [ ]:
con.execute(
    """
    CREATE TABLE core AS
    SELECT
        user_id,
        video_id,
        any_value(ccid) AS ccid,
        min("timestamp") AS "timestamp",
        max(last_timestamp) AS last_timestamp,
        max(duration_seconds) AS duration_seconds,
        least(
            max(duration_seconds),
            sum(watched_seconds)
        ) AS watched_seconds,
        sum(playback_seconds) AS playback_seconds,
        least(
            1.0,
            sum(watched_seconds) /
            nullif(max(duration_seconds), 0)
        ) AS completion_ratio,
        sum(segment_count)::INTEGER AS segment_count,
        max(engagement_weight) AS engagement_weight,
        1::TINYINT AS positive
    FROM interactions
    GROUP BY user_id, video_id
    """
)

deduplicated_rows = con.execute(
    "SELECT COUNT(*) FROM core"
).fetchone()[0]

print(f"Deduplicated user-video rows: {deduplicated_rows:,}")

In [ ]:
MIN_USER_INTERACTIONS = 5
MIN_VIDEO_USERS = 5

for iteration in range(1, 21):
    before = con.execute(
        "SELECT COUNT(*) FROM core"
    ).fetchone()[0]

    con.execute("DROP TABLE IF EXISTS core_next")

    con.execute(
        f"""
        CREATE TABLE core_next AS
        SELECT c.*
        FROM core AS c

        JOIN (
            SELECT user_id
            FROM core
            GROUP BY user_id
            HAVING COUNT(*) >= {MIN_USER_INTERACTIONS}
        ) AS eligible_users
        USING (user_id)

        JOIN (
            SELECT video_id
            FROM core
            GROUP BY video_id
            HAVING COUNT(DISTINCT user_id) >= {MIN_VIDEO_USERS}
        ) AS eligible_videos
        USING (video_id)
        """
    )

    after = con.execute(
        "SELECT COUNT(*) FROM core_next"
    ).fetchone()[0]

    con.execute("DROP TABLE core")
    con.execute("ALTER TABLE core_next RENAME TO core")

    print(
        f"Sparse-filter iteration {iteration}: "
        f"{before:,} -> {after:,}"
    )

    if after == 0:
        raise RuntimeError(
            "Sparse filtering removed every interaction."
        )

    if after == before:
        print("Sparse filtering converged.")
        break

core_statistics = con.execute(
    """
    SELECT
        COUNT(*) AS interactions,
        COUNT(DISTINCT user_id) AS users,
        COUNT(DISTINCT video_id) AS videos
    FROM core
    """
).df()

display(core_statistics)

In [ ]:
tables_to_remove = [
    "ranked",
    "train_base",
    "valid_base",
    "test_base",
    "eval_users",
    "eval_users_next",
    "train",
    "valid",
    "test",
]

for table in tables_to_remove:
    con.execute(f"DROP TABLE IF EXISTS {table}")

# Rank each user's interactions from newest to oldest.
con.execute(
    """
    CREATE TABLE ranked AS
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY
                "timestamp" DESC,
                last_timestamp DESC,
                video_id
        ) AS reverse_rank,
        COUNT(*) OVER (
            PARTITION BY user_id
        ) AS user_count
    FROM core
    """
)

# Latest interaction: test.
# Second latest: validation.
# Earlier interactions: training.
con.execute(
    """
    CREATE TABLE train_base AS
    SELECT * EXCLUDE (reverse_rank, user_count)
    FROM ranked
    WHERE reverse_rank > 2
    """
)

con.execute(
    """
    CREATE TABLE valid_base AS
    SELECT * EXCLUDE (reverse_rank, user_count)
    FROM ranked
    WHERE reverse_rank = 2
    """
)

con.execute(
    """
    CREATE TABLE test_base AS
    SELECT * EXCLUDE (reverse_rank, user_count)
    FROM ranked
    WHERE reverse_rank = 1
    """
)

# Initially retain users present in every split.
con.execute(
    """
    CREATE TABLE eval_users AS
    SELECT user_id FROM train_base
    INTERSECT
    SELECT user_id FROM valid_base
    INTERSECT
    SELECT user_id FROM test_base
    """
)

# Repeatedly enforce a training-only recommendation catalogue.
for iteration in range(1, 21):
    old_count = con.execute(
        "SELECT COUNT(*) FROM eval_users"
    ).fetchone()[0]

    for table in [
        "train",
        "valid",
        "test",
        "eval_users_next",
    ]:
        con.execute(f"DROP TABLE IF EXISTS {table}")

    con.execute(
        """
        CREATE TABLE train AS
        SELECT b.*
        FROM train_base AS b
        JOIN eval_users USING (user_id)
        """
    )

    con.execute(
        """
        CREATE TABLE valid AS
        SELECT b.*
        FROM valid_base AS b
        JOIN eval_users USING (user_id)
        JOIN (
            SELECT DISTINCT video_id
            FROM train
        ) AS training_catalog
        USING (video_id)
        """
    )

    con.execute(
        """
        CREATE TABLE test AS
        SELECT b.*
        FROM test_base AS b
        JOIN eval_users USING (user_id)
        JOIN (
            SELECT DISTINCT video_id
            FROM train
        ) AS training_catalog
        USING (video_id)
        """
    )

    con.execute(
        """
        CREATE TABLE eval_users_next AS
        SELECT user_id FROM train
        INTERSECT
        SELECT user_id FROM valid
        INTERSECT
        SELECT user_id FROM test
        """
    )

    new_count = con.execute(
        "SELECT COUNT(*) FROM eval_users_next"
    ).fetchone()[0]

    con.execute("DROP TABLE eval_users")
    con.execute(
        "ALTER TABLE eval_users_next RENAME TO eval_users"
    )

    print(
        f"Evaluation cleanup {iteration}: "
        f"{old_count:,} -> {new_count:,} users"
    )

    if new_count == 0:
        raise RuntimeError(
            "No evaluation users remain."
        )

    if new_count == old_count:
        print("Evaluation cleanup converged.")
        break

# Rebuild the three tables using the stable user set.
for table in ["train", "valid", "test"]:
    con.execute(f"DROP TABLE IF EXISTS {table}")

con.execute(
    """
    CREATE TABLE train AS
    SELECT b.*
    FROM train_base AS b
    JOIN eval_users USING (user_id)
    """
)

con.execute(
    """
    CREATE TABLE valid AS
    SELECT b.*
    FROM valid_base AS b
    JOIN eval_users USING (user_id)
    JOIN (
        SELECT DISTINCT video_id
        FROM train
    ) AS training_catalog
    USING (video_id)
    """
)

con.execute(
    """
    CREATE TABLE test AS
    SELECT b.*
    FROM test_base AS b
    JOIN eval_users USING (user_id)
    JOIN (
        SELECT DISTINCT video_id
        FROM train
    ) AS training_catalog
    USING (video_id)
    """
)

split_statistics = con.execute(
    """
    SELECT
        'train' AS split,
        COUNT(*) AS interactions,
        COUNT(DISTINCT user_id) AS users,
        COUNT(DISTINCT video_id) AS videos
    FROM train

    UNION ALL

    SELECT
        'validation',
        COUNT(*),
        COUNT(DISTINCT user_id),
        COUNT(DISTINCT video_id)
    FROM valid

    UNION ALL

    SELECT
        'test',
        COUNT(*),
        COUNT(DISTINCT user_id),
        COUNT(DISTINCT video_id)
    FROM test
    """
).df()

display(split_statistics)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

OUT = Path("/kaggle/working/processed")
SPLITS = OUT / "splits"
GRAPH = OUT / "graph"
REPORTS = OUT / "reports"

for directory in [OUT, SPLITS, GRAPH, REPORTS]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 1. Save chronological splits
# ------------------------------------------------------------------

split_files = {
    "train": SPLITS / "train.parquet",
    "valid": SPLITS / "valid.parquet",
    "test": SPLITS / "test.parquet",
}

for table, output_file in split_files.items():
    con.execute(
        f"""
        COPY (
            SELECT *
            FROM {table}
            ORDER BY user_id, "timestamp", video_id
        )
        TO '{output_file.as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        f"Saved {table}:",
        output_file,
        f"({output_file.stat().st_size / 1024**2:.2f} MB)"
    )

split_statistics.to_csv(
    REPORTS / "split_statistics.csv",
    index=False
)

# ------------------------------------------------------------------
# 2. Create user and video integer mappings
# ------------------------------------------------------------------

users = con.execute(
    """
    SELECT DISTINCT user_id
    FROM train
    ORDER BY user_id
    """
).df()

users["user_idx"] = np.arange(
    len(users),
    dtype=np.int64
)

videos = con.execute(
    """
    SELECT
        video_id,
        any_value(ccid) AS ccid
    FROM train
    GROUP BY video_id
    ORDER BY video_id
    """
).df()

videos["video_idx"] = np.arange(
    len(videos),
    dtype=np.int64
)

users.to_parquet(
    GRAPH / "user_index.parquet",
    index=False
)

videos.to_parquet(
    GRAPH / "video_index.parquet",
    index=False
)

train_video_ids = set(
    videos["video_id"].astype(str)
)

train_ccids = set(
    videos["ccid"].dropna().astype(str)
)

print("Indexed users:", len(users))
print("Indexed videos:", len(videos))

# ------------------------------------------------------------------
# 3. Build concept-video graph
# ------------------------------------------------------------------

concept_edges = []

concept_video_file = (
    META_ROOT / "relations/concept-video.txt"
)

with concept_video_file.open(
    "r",
    encoding="utf-8",
    errors="replace"
) as handle:

    for line in tqdm(
        handle,
        desc="Building concept-video graph"
    ):
        parts = line.rstrip("\n").split("\t")

        if len(parts) < 2:
            parts = line.strip().split()

        if len(parts) < 2:
            continue

        left = str(parts[0])
        right = str(parts[1])

        if left in train_ccids:
            concept_id = right
            ccid = left

        elif right in train_ccids:
            concept_id = left
            ccid = right

        else:
            continue

        concept_edges.append(
            (concept_id, ccid)
        )

concept_video_edges = pd.DataFrame(
    concept_edges,
    columns=["concept_id", "ccid"]
).drop_duplicates()

concept_ids = sorted(
    concept_video_edges["concept_id"]
    .astype(str)
    .unique()
)

concept_index = pd.DataFrame({
    "concept_id": concept_ids,
    "concept_idx": np.arange(
        len(concept_ids),
        dtype=np.int64
    )
})

concept_video_edges.to_parquet(
    GRAPH / "concept_video_edges.parquet",
    index=False
)

concept_index.to_parquet(
    GRAPH / "concept_index.parquet",
    index=False
)

print(
    "Concept-video edges:",
    len(concept_video_edges)
)

print(
    "Unique concepts:",
    len(concept_index)
)

# ------------------------------------------------------------------
# 4. Build course-video graph
# ------------------------------------------------------------------

course_rows = []

course_file = META_ROOT / "entities/course.json"

for course in tqdm(
    stream_json(course_file),
    desc="Building course-video graph"
):
    course_id = str(
        course.get("id")
        or course.get("course_id")
        or ""
    )

    for resource in course.get("resource") or []:
        resource_id = str(
            resource.get("resource_id") or ""
        )

        ccid = video_to_ccid.get(resource_id)

        if ccid not in train_ccids:
            continue

        titles = resource.get("titles") or []

        course_rows.append({
            "course_id": course_id,
            "course_video_id": resource_id,
            "ccid": ccid,
            "chapter": str(
                resource.get("chapter") or ""
            ),
            "titles": " / ".join(
                str(value)
                for value in titles
                if str(value).strip()
            )
        })

course_video_edges = pd.DataFrame(
    course_rows,
    columns=[
        "course_id",
        "course_video_id",
        "ccid",
        "chapter",
        "titles",
    ]
).drop_duplicates(
    ["course_id", "ccid"]
)

course_video_edges.to_parquet(
    GRAPH / "course_video_edges.parquet",
    index=False
)

print(
    "Course-video edges:",
    len(course_video_edges)
)

# ------------------------------------------------------------------
# 5. Save video captions and metadata
# ------------------------------------------------------------------

video_metadata_rows = []

for video_id in sorted(train_video_ids):
    ccid = video_to_ccid.get(video_id)
    metadata = metadata_by_ccid.get(ccid)

    if metadata is None:
        continue

    video_metadata_rows.append({
        "video_id": video_id,
        **metadata
    })

video_metadata = pd.DataFrame(
    video_metadata_rows,
    columns=[
        "video_id",
        "ccid",
        "duration_seconds",
        "caption_segment_count",
        "caption_text",
        "video_name",
        "start_segment_count",
    ]
)

video_metadata.to_parquet(
    GRAPH / "video_metadata.parquet",
    index=False
)

print(
    "Videos with caption metadata:",
    len(video_metadata)
)

# ------------------------------------------------------------------
# 6. Save concept names
# ------------------------------------------------------------------

wanted_concepts = set(concept_ids)
concept_metadata_rows = []

concept_file = META_ROOT / "entities/concept.json"

for record in tqdm(
    stream_json(concept_file),
    desc="Reading concept metadata"
):
    concept_id = str(
        record.get("id") or ""
    )

    if concept_id not in wanted_concepts:
        continue

    concept_metadata_rows.append({
        "concept_id": concept_id,
        "concept_name": str(
            record.get("name") or ""
        ),
        "context_count": len(
            record.get("context") or []
        )
    })

concept_metadata = pd.DataFrame(
    concept_metadata_rows,
    columns=[
        "concept_id",
        "concept_name",
        "context_count",
    ]
)

concept_metadata.to_parquet(
    GRAPH / "concept_metadata.parquet",
    index=False
)

print(
    "Concept metadata rows:",
    len(concept_metadata)
)

# ------------------------------------------------------------------
# 7. Leakage and chronology validation
# ------------------------------------------------------------------

checks = {
    "train_is_nonempty":
        con.execute(
            "SELECT COUNT(*) > 0 FROM train"
        ).fetchone()[0],

    "validation_is_nonempty":
        con.execute(
            "SELECT COUNT(*) > 0 FROM valid"
        ).fetchone()[0],

    "test_is_nonempty":
        con.execute(
            "SELECT COUNT(*) > 0 FROM test"
        ).fetchone()[0],

    "train_before_or_at_validation":
        con.execute(
            """
            SELECT COUNT(*) = 0
            FROM (
                SELECT
                    user_id,
                    MAX("timestamp") AS maximum_time
                FROM train
                GROUP BY user_id
            ) AS training_time
            JOIN valid USING (user_id)
            WHERE
                training_time.maximum_time >
                valid."timestamp"
            """
        ).fetchone()[0],

    "validation_before_or_at_test":
        con.execute(
            """
            SELECT COUNT(*) = 0
            FROM valid
            JOIN test USING (user_id)
            WHERE
                valid."timestamp" >
                test."timestamp"
            """
        ).fetchone()[0],

    "validation_items_in_training_catalog":
        con.execute(
            """
            SELECT COUNT(*) = 0
            FROM valid
            WHERE NOT EXISTS (
                SELECT 1
                FROM train
                WHERE
                    train.video_id =
                    valid.video_id
            )
            """
        ).fetchone()[0],

    "test_items_in_training_catalog":
        con.execute(
            """
            SELECT COUNT(*) = 0
            FROM test
            WHERE NOT EXISTS (
                SELECT 1
                FROM train
                WHERE
                    train.video_id =
                    test.video_id
            )
            """
        ).fetchone()[0],

    "same_validation_and_test_users":
        con.execute(
            """
            SELECT COUNT(*) = 0
            FROM (
                (
                    SELECT user_id FROM valid
                    EXCEPT
                    SELECT user_id FROM test
                )
                UNION ALL
                (
                    SELECT user_id FROM test
                    EXCEPT
                    SELECT user_id FROM valid
                )
            )
            """
        ).fetchone()[0],

    "unique_user_video_pairs":
        con.execute(
            """
            SELECT
                (
                    SELECT COUNT(*) FROM train
                ) = (
                    SELECT COUNT(*)
                    FROM (
                        SELECT DISTINCT
                            user_id,
                            video_id
                        FROM train
                    )
                )
                AND
                (
                    SELECT COUNT(*) FROM valid
                ) = (
                    SELECT COUNT(*)
                    FROM (
                        SELECT DISTINCT
                            user_id,
                            video_id
                        FROM valid
                    )
                )
                AND
                (
                    SELECT COUNT(*) FROM test
                ) = (
                    SELECT COUNT(*)
                    FROM (
                        SELECT DISTINCT
                            user_id,
                            video_id
                        FROM test
                    )
                )
            """
        ).fetchone()[0],
}

validation = pd.DataFrame(
    checks.items(),
    columns=["check", "passed"]
)

validation.to_csv(
    REPORTS / "preprocessing_validation.csv",
    index=False
)

display(validation)

if not validation["passed"].astype(bool).all():
    failed = validation.loc[
        ~validation["passed"].astype(bool)
    ]

    raise AssertionError(
        "Preprocessing validation failed:\n"
        + failed.to_string(index=False)
    )

print("All preprocessing checks passed.")

# ------------------------------------------------------------------
# 8. Save preprocessing manifest
# ------------------------------------------------------------------

manifest = {
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_file": str(USER_VIDEO),

    "source_size_bytes":
        USER_VIDEO.stat().st_size,

    "parameters": {
        "minimum_user_interactions":
            MIN_USER_INTERACTIONS,

        "minimum_video_users":
            MIN_VIDEO_USERS,

        "minimum_watch_seconds":
            MIN_WATCH_SECONDS,

        "minimum_completion_ratio":
            MIN_COMPLETION_RATIO,

        "maximum_video_duration_seconds":
            SHORT_VIDEO_MAX_SECONDS,
    },

    "split_method":
        "Per-user chronological leave-two-out",

    "split_statistics":
        split_statistics.to_dict(
            orient="records"
        ),

    "graph_statistics": {
        "users": len(users),
        "videos": len(videos),
        "concepts": len(concept_index),
        "concept_video_edges":
            len(concept_video_edges),
        "course_video_edges":
            len(course_video_edges),
        "videos_with_caption_metadata":
            len(video_metadata),
    },

    "files": sorted(
        str(path.relative_to(OUT))
        for path in OUT.rglob("*")
        if path.is_file()
    ),
}

with (
    REPORTS / "preprocessing_manifest.json"
).open(
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        manifest,
        handle,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------------
# 9. Create downloadable ZIP
# ------------------------------------------------------------------

zip_path = shutil.make_archive(
    "/kaggle/working/MOOCCubeX_processed",
    "zip",
    root_dir=str(OUT.parent),
    base_dir=OUT.name,
)

print("\nPreprocessing completed successfully.")
print("Processed data:", OUT)
print("Downloadable ZIP:", zip_path)
print(
    "ZIP size:",
    f"{Path(zip_path).stat().st_size / 1024**2:.2f} MB"
)

display(split_statistics)

In [ ]:
"""Three-seed validation-tuned BCE-SASRec experiment for Kaggle.

Requires preprocessed MOOCCubeX splits and graph Parquet files under /kaggle/input.
Hyperparameters are chosen only by validation NDCG@10. The test set is evaluated
after selection for seeds 42, 2026 and 3407. A target is reported, never forced.
"""
import warnings
warnings.filterwarnings("ignore", message="enable_nested_tensor is True")
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)


# ---- notebook cell 3 ----

from pathlib import Path
from dataclasses import dataclass, asdict
import copy, gc, json, math, os, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

@dataclass
class Config:
    root: str = '/kaggle/input'
    seed: int = 42
    max_len: int = 50
    max_concepts_per_video: int = 12
    hidden_dim: int = 128
    transformer_layers: int = 2
    attention_heads: int = 4
    feedforward_dim: int = 512
    dropout: float = 0.10
    time_buckets: int = 32
    negatives: int = 50
    batch_size: int = 128
    eval_batch_size: int = 128
    max_epochs: int = 25
    minimum_epochs: int = 10
    early_stopping_patience: int = 5
    early_stopping_min_delta: float = 1e-4
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    concept_loss_weight: float = 0.20
    completion_loss_weight: float = 0.10
    gradient_clip: float = 5.0
    use_mixed_precision: bool = False
    ks: tuple = (5, 10, 20)
    num_workers: int = 2

CFG=Config()

# Prefer preprocessing outputs created in this Kaggle session.
working_processed=Path('/kaggle/working/processed')
search_roots=[working_processed,Path('/kaggle/input')]
matches=[]
for search_root in search_roots:
    if search_root.name=='processed':
        candidates=[search_root/'splits/train.parquet']
    else:
        candidates=[p for p in search_root.rglob('train.parquet') if p.parent.name=='splits']
    for train_path in candidates:
        processed=train_path.parent.parent
        graph=processed/'graph'
        required_graph=['video_metadata.parquet','concept_video_edges.parquet','video_index.parquet','course_video_edges.parquet']
        if train_path.exists() and all((graph/name).exists() for name in required_graph) and (train_path.parent/'valid.parquet').exists() and (train_path.parent/'test.parquet').exists():
            matches.append(processed)
    if matches:break
if not matches:
    raise FileNotFoundError('Could not find a complete processed/splits and processed/graph dataset.')
PROCESSED=matches[0];SPLITS=PROCESSED/'splits';GRAPH=PROCESSED/'graph';ROOT=PROCESSED.parent

# Kaggle inputs are read-only; every generated artifact goes to /kaggle/working.
OUT=Path('/kaggle/working/tuned_bce_sasrec');CHECKPOINTS=OUT/'checkpoints';REPORTS=OUT/'reports';EXPLANATIONS=OUT/'explanations'
for p in [OUT,CHECKPOINTS,REPORTS,EXPLANATIONS]:p.mkdir(parents=True,exist_ok=True)
print('Detected processed data:',PROCESSED)
print('Output directory:',OUT)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

seed_everything(CFG.seed)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print(json.dumps(asdict(CFG),indent=2))

# ---- notebook cell 5 ----

required=[SPLITS/'train.parquet',SPLITS/'valid.parquet',SPLITS/'test.parquet',
          GRAPH/'video_metadata.parquet',GRAPH/'concept_video_edges.parquet',
          GRAPH/'video_index.parquet',GRAPH/'course_video_edges.parquet']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError(f'Missing preprocessing outputs: {missing}')

train_df=pd.read_parquet(SPLITS/'train.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
valid_df=pd.read_parquet(SPLITS/'valid.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
test_df=pd.read_parquet(SPLITS/'test.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)

required_columns={'user_id','video_id','timestamp','duration_seconds','watched_seconds',
                  'playback_seconds','completion_ratio','segment_count','engagement_weight'}
for name,frame in [('train',train_df),('valid',valid_df),('test',test_df)]:
    absent=required_columns-set(frame.columns)
    if absent: raise ValueError(f'{name} is missing columns: {sorted(absent)}')
    frame['user_id']=frame.user_id.astype(str); frame['video_id']=frame.video_id.astype(str)
    frame['timestamp']=pd.to_numeric(frame.timestamp,errors='coerce').fillna(0).astype('int64')

item_ids=sorted(train_df.video_id.unique()); user_ids=sorted(set(train_df.user_id)|set(valid_df.user_id)|set(test_df.user_id))
item2idx={v:i+1 for i,v in enumerate(item_ids)}; idx2item={i:v for v,i in item2idx.items()}
user2idx={u:i for i,u in enumerate(user_ids)}; idx2user={i:u for u,i in user2idx.items()}
num_items,num_users=len(item2idx),len(user2idx)

def map_frame(frame):
    x=frame[frame.video_id.isin(item2idx)].copy().reset_index(drop=True)
    x['u']=x.user_id.map(user2idx).astype('int64'); x['i']=x.video_id.map(item2idx).astype('int64')
    return x

train=map_frame(train_df);valid=map_frame(valid_df);test=map_frame(test_df)
print({'train':len(train),'validation':len(valid),'test':len(test),'users':num_users,'videos':num_items})
display(pd.DataFrame([
    ['train',len(train),train.u.nunique(),train.i.nunique()],
    ['validation',len(valid),valid.u.nunique(),valid.i.nunique()],
    ['test',len(test),test.u.nunique(),test.i.nunique()]],
    columns=['split','interactions','users','videos']))

# ---- notebook cell 7 ----

train_max=train.groupby('u').timestamp.max()
valid_by_user=valid.set_index('u'); test_by_user=test.set_index('u')
common_users=sorted(set(train_max.index)&set(valid_by_user.index)&set(test_by_user.index))

checks={
 'train_before_or_at_validation':bool((train_max.loc[common_users].values<=valid_by_user.loc[common_users].timestamp.values).all()),
 'validation_before_or_at_test':bool((valid_by_user.loc[common_users].timestamp.values<=test_by_user.loc[common_users].timestamp.values).all()),
 'validation_items_in_training_catalog':bool(valid.i.isin(set(train.i)).all()),
 'test_items_in_training_catalog':bool(test.i.isin(set(train.i)).all()),
 'same_validation_and_test_users':set(valid.u)==set(test.u),
}
display(pd.DataFrame(checks.items(),columns=['check','passed']))
if not all(checks.values()): raise AssertionError('Leakage/chronology audit failed.')

# ---- notebook cell 9 ----

BEHAVIOUR_COLUMNS=['watched_seconds','playback_seconds','duration_seconds',
                   'completion_ratio','segment_count','engagement_weight']
LOG_BEHAVIOUR={'watched_seconds','playback_seconds','duration_seconds','segment_count'}

def raw_behaviour(frame):
    x=frame[BEHAVIOUR_COLUMNS].astype('float32').replace([np.inf,-np.inf],np.nan).fillna(0).copy()
    for c in LOG_BEHAVIOUR:x[c]=np.log1p(x[c].clip(lower=0))
    return x

train_beh_raw=raw_behaviour(train);beh_mean=train_beh_raw.mean();beh_std=train_beh_raw.std().replace(0,1).fillna(1)
def normalized_behaviour(frame):return ((raw_behaviour(frame)-beh_mean)/beh_std).astype('float32').to_numpy()
train_beh=normalized_behaviour(train);valid_beh=normalized_behaviour(valid);test_beh=normalized_behaviour(test)
normalization={'columns':BEHAVIOUR_COLUMNS,'mean':beh_mean.to_dict(),'std':beh_std.to_dict(),'log1p_columns':sorted(LOG_BEHAVIOUR)}
json.dump(normalization,open(REPORTS/'behaviour_normalization.json','w'),indent=2)
display(pd.DataFrame({'mean':beh_mean,'std':beh_std}))

# ---- notebook cell 11 ----

video_index=pd.read_parquet(GRAPH/'video_index.parquet');video_index['video_id']=video_index.video_id.astype(str);video_index['ccid']=video_index.ccid.astype(str)
video_to_ccid=dict(zip(video_index.video_id,video_index.ccid));ccid_to_item={video_to_ccid[v]:item2idx[v] for v in item2idx if v in video_to_ccid}

cv=pd.read_parquet(GRAPH/'concept_video_edges.parquet');cv['concept_id']=cv.concept_id.astype(str);cv['ccid']=cv.ccid.astype(str)
cv=cv[cv.ccid.isin(ccid_to_item)].drop_duplicates(['ccid','concept_id'])
concept_ids=sorted(cv.concept_id.unique());concept2idx={c:i+1 for i,c in enumerate(concept_ids)};idx2concept={i:c for c,i in concept2idx.items()}
item_concepts=np.zeros((num_items+1,CFG.max_concepts_per_video),dtype=np.int64)
for ccid,g in cv.groupby('ccid'):
    ids=[concept2idx[c] for c in g.concept_id.iloc[:CFG.max_concepts_per_video]]
    item_concepts[ccid_to_item[ccid],:len(ids)]=ids

course=pd.read_parquet(GRAPH/'course_video_edges.parquet')
if 'video_id' not in course.columns:
    if 'course_video_id' in course.columns:course=course.rename(columns={'course_video_id':'video_id'})
    else:raise ValueError('course_video_edges.parquet needs video_id or course_video_id')
course['video_id']=course.video_id.astype(str);course['course_id']=course.course_id.astype(str)
course=course[course.video_id.isin(item2idx)].drop_duplicates('video_id')
course_ids=sorted(course.course_id.unique());course2idx={c:i+1 for i,c in enumerate(course_ids)}
item_course=np.zeros(num_items+1,dtype=np.int64)
for row in course.itertuples():item_course[item2idx[row.video_id]]=course2idx[row.course_id]

metadata=pd.read_parquet(GRAPH/'video_metadata.parquet');metadata['video_id']=metadata.video_id.astype(str)
META_COLUMNS=['duration_seconds','subtitle_sentences','subtitle_characters','concept_count']
for c in META_COLUMNS:
    if c not in metadata:metadata[c]=0
metadata=metadata.drop_duplicates('video_id').set_index('video_id')
meta_table=pd.DataFrame(index=item_ids,columns=META_COLUMNS,dtype='float32')
for c in META_COLUMNS:meta_table[c]=pd.to_numeric(metadata.reindex(item_ids)[c],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
for c in META_COLUMNS:meta_table[c]=np.log1p(meta_table[c].clip(lower=0))
meta_mean=meta_table.mean();meta_std=meta_table.std().replace(0,1).fillna(1)
meta_table=((meta_table-meta_mean)/meta_std).astype('float32')
item_metadata=np.zeros((num_items+1,len(META_COLUMNS)),dtype=np.float32);item_metadata[1:]=meta_table.to_numpy()

print({'concepts':len(concept_ids),'concept_video_edges':len(cv),'courses':len(course_ids),
       'videos_with_concepts':int((item_concepts!=0).any(1).sum()),'videos_with_courses':int((item_course!=0).sum())})
json.dump({'meta_columns':META_COLUMNS,'mean':meta_mean.to_dict(),'std':meta_std.to_dict()},open(REPORTS/'metadata_normalization.json','w'),indent=2)

# ---- notebook cell 13 ----

def build_history(frame,features):
    out={}
    for u,idx in frame.groupby('u',sort=False).groups.items():
        rows=np.asarray(list(idx));g=frame.loc[rows]
        out[int(u)]={'items':g.i.astype(int).tolist(),'times':g.timestamp.astype('int64').tolist(),
                     'behaviour':features[rows].tolist(),'completion':g.completion_ratio.astype(float).tolist()}
    return out

train_h=build_history(train,train_beh);valid_events=build_history(valid,valid_beh);test_events=build_history(test,test_beh)
eval_users=sorted(set(train_h)&set(valid_events)&set(test_events))
valid_hist={u:copy.deepcopy(train_h[u]) for u in eval_users}
test_hist={}
valid_target={u:valid_events[u]['items'][0] for u in eval_users};test_target={u:test_events[u]['items'][0] for u in eval_users}
valid_completion={u:valid_events[u]['completion'][0] for u in eval_users};test_completion={u:test_events[u]['completion'][0] for u in eval_users}
for u in eval_users:
    h=copy.deepcopy(train_h[u])
    for key in ['items','times','behaviour','completion']:h[key].append(valid_events[u][key][0])
    test_hist[u]=h

all_positive={u:set(train_h[u]['items'])|{valid_target[u],test_target[u]} for u in eval_users}
for u in train_h:
    all_positive.setdefault(u,set(train_h[u]['items']))
assert all(valid_target[u] not in valid_hist[u]['items'] for u in eval_users)
assert all(test_target[u] not in test_hist[u]['items'] for u in eval_users)
print('Leakage-free evaluation users:',len(eval_users))

# ---- notebook cell 15 ----

def left_pad(seq,n,pad):
    # Right padding prevents fully masked attention rows under a causal mask.
    seq=list(seq)[-n:];return seq+[pad]*(n-len(seq))

class PrefixDataset(Dataset):
    def __init__(self,histories,max_len):
        self.h=histories;self.max_len=max_len
        self.examples=[(u,t) for u,h in histories.items() for t in range(1,len(h['items']))]
    def __len__(self):return len(self.examples)
    def __getitem__(self,index):
        u,t=self.examples[index];h=self.h[u]
        return (torch.tensor(u),torch.tensor(left_pad(h['items'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['times'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['behaviour'][:t],self.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)),dtype=torch.float32),
                torch.tensor(h['items'][t]),torch.tensor(h['completion'][t],dtype=torch.float32))

train_dataset=PrefixDataset(train_h,CFG.max_len)
train_loader=DataLoader(train_dataset,batch_size=CFG.batch_size,shuffle=True,num_workers=CFG.num_workers,
                        pin_memory=True,persistent_workers=CFG.num_workers>0)
print('Training prefix examples:',len(train_dataset),'batches per epoch:',len(train_loader))

def sample_negatives(users,count):
    result=[]
    for u in users.tolist():
        values=[];known=all_positive[int(u)]
        while len(values)<count:
            x=random.randint(1,num_items)
            if x not in known:values.append(x)
        result.append(values)
    return torch.tensor(result,dtype=torch.long)

# ---- notebook cell 17 ----

class BCESASRec(nn.Module):
    def __init__(self,num_items,num_concepts,num_courses,item_concepts,item_course,item_metadata,cfg):
        super().__init__();self.cfg=cfg;d=cfg.hidden_dim
        self.item_emb=nn.Embedding(num_items+1,d,padding_idx=0)
        self.concept_emb=nn.Embedding(num_concepts+1,d,padding_idx=0)
        self.course_emb=nn.Embedding(num_courses+1,d,padding_idx=0)
        self.position_emb=nn.Embedding(cfg.max_len,d);self.time_emb=nn.Embedding(cfg.time_buckets,d,padding_idx=0)
        self.behaviour_mlp=nn.Sequential(nn.Linear(len(BEHAVIOUR_COLUMNS),64),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(64,d))
        self.metadata_mlp=nn.Sequential(nn.Linear(len(META_COLUMNS),64),nn.GELU(),nn.Linear(64,d))
        self.concept_query=nn.Linear(d,d,bias=False);self.concept_key=nn.Linear(d,d,bias=False)
        self.event_norm=nn.LayerNorm(d);self.candidate_norm=nn.LayerNorm(d);self.dropout=nn.Dropout(cfg.dropout)
        layer=nn.TransformerEncoderLayer(d,cfg.attention_heads,cfg.feedforward_dim,cfg.dropout,
                batch_first=True,norm_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(layer,cfg.transformer_layers);self.output_norm=nn.LayerNorm(d)
        self.completion_head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(d,1))
        self.register_buffer('item_concepts',torch.tensor(item_concepts,dtype=torch.long))
        self.register_buffer('item_course',torch.tensor(item_course,dtype=torch.long))
        self.register_buffer('item_metadata',torch.tensor(item_metadata,dtype=torch.float32))
        self.scale=math.sqrt(d)

    def concept_pool(self,item_idx,return_weights=False):
        ids=self.item_concepts[item_idx];c=self.concept_emb(ids);q=self.concept_query(self.item_emb(item_idx)).unsqueeze(-2)
        # Calculate masking and normalization in FP32. In FP16, 1e-8 becomes zero;
        # videos with no concepts would therefore divide 0 by 0 and produce NaN.
        logits=((q*self.concept_key(c)).sum(-1)/self.scale).float();mask=ids.eq(0)
        logits=logits.masked_fill(mask,-1e9);weights=torch.softmax(logits,dim=-1)
        weights=weights.masked_fill(mask,0.0)
        weights=weights/weights.sum(-1,keepdim=True).clamp_min(1.0)
        pooled=(weights.unsqueeze(-1)*c.float()).sum(-2).to(c.dtype)
        return (pooled,weights,ids) if return_weights else pooled

    def candidate(self,item_idx):
        z=self.item_emb(item_idx)+self.concept_pool(item_idx)+self.course_emb(self.item_course[item_idx])+self.metadata_mlp(self.item_metadata[item_idx])
        return self.candidate_norm(z)

    def time_bucket(self,times):
        gap=torch.zeros_like(times);valid=(times[:,1:]>0)&(times[:,:-1]>0)
        delta=(times[:,1:]-times[:,:-1]).clamp_min(0)
        gap[:,1:]=torch.where(valid,delta,torch.zeros_like(delta))
        bucket=torch.floor(torch.log2(gap.float()+1)).long()+1
        return bucket.clamp(0,self.cfg.time_buckets-1).masked_fill(times.eq(0),0)

    def encode(self,seq,times,behaviour):
        pos=torch.arange(self.cfg.max_len,device=seq.device).unsqueeze(0)
        static=self.candidate(seq);x=static+self.behaviour_mlp(behaviour)+self.time_emb(self.time_bucket(times))+self.position_emb(pos)
        padding=seq.eq(0);x=self.dropout(self.event_norm(x));x=x.masked_fill(padding.unsqueeze(-1),0.0)
        causal=torch.triu(torch.ones(self.cfg.max_len,self.cfg.max_len,device=seq.device,dtype=torch.bool),1)
        x=self.transformer(x,mask=causal,src_key_padding_mask=padding)
        last_index=seq.ne(0).sum(1).clamp_min(1)-1
        last=x[torch.arange(len(seq),device=seq.device),last_index]
        return self.output_norm(last)

    def sampled_logits(self,h,candidates):return (h.unsqueeze(1)*self.candidate(candidates)).sum(-1)/self.scale
    def all_candidate_embeddings(self):return self.candidate(torch.arange(1,self.item_emb.num_embeddings,device=self.item_emb.weight.device))
    def completion(self,h,item):return torch.sigmoid(self.completion_head(torch.cat([h,self.candidate(item)],-1))).squeeze(-1)

model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,CFG).to(device)
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))

# ---- notebook cell 19 ----

def calculate_metrics(ranks,topk_items,item_popularity,num_items,ks=(5,10,20)):
    ranks=np.asarray(ranks,dtype=np.int64);out={'Accuracy@1':float(np.mean(ranks==1)),'MRR':float(np.mean(1/ranks)),
        'MeanRank':float(np.mean(ranks)),'MedianRank':float(np.median(ranks))}
    for k in ks:
        hit=ranks<=k;recall=float(hit.mean());precision=recall/k
        out[f'Precision@{k}']=precision;out[f'Recall@{k}']=recall
        out[f'F1@{k}']=0.0 if recall==0 else float(2*precision*recall/(precision+recall))
        out[f'NDCG@{k}']=float(np.mean(np.where(hit,1/np.log2(ranks+1),0)))
        out[f'MAP@{k}']=float(np.mean(np.where(hit,1/ranks,0)))
    rec=np.asarray(topk_items);out['CatalogCoverage@10']=float(len(np.unique(rec))/num_items)
    total=sum(item_popularity.values());probs=np.array([item_popularity.get(int(i),0.5)/total for i in rec.ravel()])
    out['Novelty@10']=float(np.mean(-np.log2(np.clip(probs,1e-12,None))))
    return out

item_popularity=train.i.value_counts().to_dict()

def tensors_for_users(histories,users):
    seq=torch.tensor([left_pad(histories[u]['items'],CFG.max_len,0) for u in users],device=device)
    times=torch.tensor([left_pad(histories[u]['times'],CFG.max_len,0) for u in users],device=device)
    beh=torch.tensor([left_pad(histories[u]['behaviour'],CFG.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)) for u in users],dtype=torch.float32,device=device)
    return seq,times,beh

@torch.no_grad()
def evaluate(model,histories,targets,target_completion,users,description):
    model.eval();candidate_z=model.all_candidate_embeddings();ranks=[];top_items=[];loss_sum=0.;n=0;completion_errors=[]
    for start in tqdm(range(0,len(users),CFG.eval_batch_size),desc=description,leave=False):
        us=users[start:start+CFG.eval_batch_size];seq,times,beh=tensors_for_users(histories,us)
        h=model.encode(seq,times,beh);scores=(h@candidate_z.T)/model.scale
        if not torch.isfinite(scores).all():
            raise FloatingPointError('Non-finite evaluation scores detected; metrics were not calculated.')
        target=torch.tensor([targets[u]-1 for u in us],device=device)
        for row,u in enumerate(us):
            seen=set(histories[u]['items']);seen.discard(targets[u])
            if seen:scores[row,torch.tensor([i-1 for i in seen],device=device)]=torch.finfo(scores.dtype).min
        loss_sum+=F.cross_entropy(scores,target,reduction='sum').item();n+=len(us)
        target_scores=scores[torch.arange(len(us),device=device),target]
        ranks.extend(((scores>target_scores.unsqueeze(1)).sum(1)+1).cpu().tolist())
        top_items.extend((torch.topk(scores,k=10,dim=1).indices+1).cpu().tolist())
        completion_pred=model.completion(h,target+1).float()
        if not torch.isfinite(completion_pred).all():
            raise FloatingPointError('Non-finite completion predictions detected.')
        completion_pred=completion_pred.cpu().numpy()
        completion_true=np.asarray([target_completion[u] for u in us],dtype=np.float32)
        completion_errors.extend((completion_pred-completion_true).tolist())
    metrics=calculate_metrics(ranks,top_items,item_popularity,num_items,CFG.ks);metrics['Loss']=loss_sum/n
    err=np.asarray(completion_errors);metrics['CompletionMAE']=float(np.mean(np.abs(err)));metrics['CompletionRMSE']=float(np.sqrt(np.mean(err**2)))
    concept_recalls=[]
    for u,recs in zip(users,top_items):
        true=set(item_concepts[targets[u]])-{0};pred=set(item_concepts[np.asarray(recs)].ravel())-{0}
        if true:concept_recalls.append(len(true&pred)/len(true))
    metrics['ConceptRecall@10']=float(np.mean(concept_recalls)) if concept_recalls else float('nan')
    return metrics,ranks,top_items

# ---- notebook cell 21 ----

train_eval_users=sorted(u for u,h in train_h.items() if len(h['items'])>=2)
train_eval_hist={};train_eval_target={};train_eval_completion={}
for u in train_eval_users:
    h=train_h[u]
    train_eval_hist[u]={k:list(v[:-1]) for k,v in h.items()}
    train_eval_target[u]=h['items'][-1]
    train_eval_completion[u]=h['completion'][-1]
print({'training_evaluation_users':len(train_eval_users),'validation_users':len(eval_users),'test_users':len(eval_users)})

# ---- notebook cell 23 ----

def concept_pair_loss(model,h,pos_items):
    ids=model.item_concepts[pos_items];mask=ids.ne(0);has=mask.any(1)
    if not has.any():return h.sum()*0
    first=mask.float().argmax(1);positive=ids[torch.arange(len(ids),device=device),first]
    negative=torch.randint(1,model.concept_emb.num_embeddings,(len(ids),),device=device)
    negative=torch.where(negative.eq(positive),(negative%(model.concept_emb.num_embeddings-1))+1,negative)
    ps=(h*model.concept_emb(positive)).sum(-1)/model.scale;ns=(h*model.concept_emb(negative)).sum(-1)/model.scale
    return -F.logsigmoid(ps[has]-ns[has]).mean()

def fit_model(cfg,name,max_epochs,trial=None):
    seed_everything(cfg.seed)
    model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,cfg).to(device)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=cfg.use_mixed_precision and device.type=='cuda')
    best=-float('inf');bad=0;history=[];path=CHECKPOINTS/f'{name}_best.pt'
    for epoch in range(1,max_epochs+1):
        started=time.time();model.train();sums=defaultdict(float);examples=0
        for users,seq,times,behaviour,pos,completion in tqdm(train_loader,desc=f'{name} {epoch:02d}/{max_epochs}',leave=False):
            users,seq,times,behaviour,pos,completion=[x.to(device,non_blocking=True) for x in [users,seq,times,behaviour,pos,completion]]
            negatives=sample_negatives(users.cpu(),cfg.negatives).to(device);candidates=torch.cat([pos[:,None],negatives],1)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type,enabled=cfg.use_mixed_precision and device.type=='cuda'):
                h=model.encode(seq,times,behaviour);logits=model.sampled_logits(h,candidates)
                ranking=F.cross_entropy(logits,torch.zeros(len(seq),dtype=torch.long,device=device),label_smoothing=.03)
                concept=concept_pair_loss(model,h,pos);pred=model.completion(h,pos)
                completion_loss=F.mse_loss(pred,completion.clamp(0,1))
                total=ranking+cfg.concept_loss_weight*concept+cfg.completion_loss_weight*completion_loss
            if not torch.isfinite(total):raise FloatingPointError(f'Non-finite loss: {name}, epoch {epoch}')
            scaler.scale(total).backward();scaler.unscale_(optimizer)
            grad=nn.utils.clip_grad_norm_(model.parameters(),cfg.gradient_clip)
            if not torch.isfinite(grad):raise FloatingPointError(f'Non-finite gradient: {name}, epoch {epoch}')
            scaler.step(optimizer);scaler.update();bs=len(seq);examples+=bs
            for k,v in [('TrainLoss',total),('RankingLoss',ranking),('ConceptLoss',concept),('CompletionLoss',completion_loss)]:sums[k]+=float(v.detach())*bs
        val,_,_=evaluate(model,valid_hist,valid_target,valid_completion,eval_users,'Validation')
        row={'Epoch':epoch,**{k:v/examples for k,v in sums.items()},**{f'Val_{k}':v for k,v in val.items()},'Seconds':time.time()-started}
        history.append(row);score=val['NDCG@10']
        print(f"{name} epoch {epoch}: loss={row['TrainLoss']:.4f}, val NDCG@10={score:.4f}, val Recall@10={val['Recall@10']:.4f}")
        if score>best+cfg.early_stopping_min_delta:
            best=score;bad=0;torch.save({'model_state':model.state_dict(),'epoch':epoch,'validation_metrics':val,'config':asdict(cfg)},path)
        else:bad+=1
        if trial is not None:
            trial.report(score,epoch)
            if trial.should_prune():raise optuna.TrialPruned()
        if epoch>=cfg.minimum_epochs and bad>=cfg.early_stopping_patience:break
    pd.DataFrame(history).to_csv(REPORTS/f'{name}_epochs.csv',index=False)
    saved=torch.load(path,map_location=device);model.load_state_dict(saved['model_state'])
    return model,saved,pd.DataFrame(history)


# ============================================================
# THREE-SEED HYPERPARAMETER EXPERIMENT
# ============================================================
try:
    import optuna
except ImportError:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','install','-q','optuna'])
    import optuna

SEEDS=[42,2026,3407]
N_TRIALS=12
TUNING_EPOCHS=10
FINAL_EPOCHS=30
TARGET_TEST_NDCG=0.90

def objective(trial):
    cfg=copy.deepcopy(CFG)
    cfg.seed=42
    cfg.hidden_dim=trial.suggest_categorical('hidden_dim',[128,256])
    cfg.transformer_layers=trial.suggest_categorical('transformer_layers',[2,3,4])
    cfg.attention_heads=trial.suggest_categorical('attention_heads',[4,8])
    cfg.feedforward_dim=trial.suggest_categorical('feedforward_dim',[512,768,1024])
    cfg.dropout=trial.suggest_float('dropout',0.08,0.25)
    cfg.learning_rate=trial.suggest_categorical('learning_rate',[1e-4,2e-4,3e-4,5e-4])
    cfg.weight_decay=trial.suggest_categorical('weight_decay',[1e-6,1e-5,1e-4])
    cfg.negatives=trial.suggest_categorical('negatives',[100,200,300])
    cfg.concept_loss_weight=trial.suggest_float('concept_loss_weight',0.08,0.30)
    cfg.completion_loss_weight=trial.suggest_categorical('completion_loss_weight',[0.02,0.05,0.10])
    cfg.gradient_clip=trial.suggest_categorical('gradient_clip',[0.5,1.0,2.0])
    cfg.minimum_epochs=5;cfg.early_stopping_patience=3
    model,saved,_=fit_model(cfg,f'hpo_trial_{trial.number:02d}',TUNING_EPOCHS,trial)
    score=float(saved['validation_metrics']['NDCG@10'])
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return score

study=optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42,multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4,n_warmup_steps=4))
study.optimize(objective,n_trials=N_TRIALS,gc_after_trial=True,show_progress_bar=True)
trials=study.trials_dataframe();trials.to_csv(REPORTS/'three_seed_hpo_trials.csv',index=False)
print('Best validation NDCG@10 during tuning:',study.best_value)
print('Best parameters:',study.best_params)

all_rows=[];histories={};manifests={}
for seed in SEEDS:
    cfg=copy.deepcopy(CFG)
    for key,value in study.best_params.items():setattr(cfg,key,value)
    cfg.seed=seed;cfg.max_epochs=FINAL_EPOCHS;cfg.minimum_epochs=10
    cfg.early_stopping_patience=6;cfg.early_stopping_min_delta=1e-4
    name=f'Tuned_BCE_SASRec_seed_{seed}'
    model,saved,history=fit_model(cfg,name,FINAL_EPOCHS)
    histories[seed]=history
    train_m,_,_=evaluate(model,train_eval_hist,train_eval_target,train_eval_completion,train_eval_users,f'Train seed {seed}')
    valid_m,_,_=evaluate(model,valid_hist,valid_target,valid_completion,eval_users,f'Validation seed {seed}')
    test_m,_,_=evaluate(model,test_hist,test_target,test_completion,eval_users,f'Test seed {seed}')
    for split,metrics in [('Train',train_m),('Validation',valid_m),('Test',test_m)]:
        all_rows.append({'Seed':seed,'Split':split,'BestEpoch':saved['epoch'],**metrics})
    manifests[str(seed)]={'best_epoch':saved['epoch'],'validation_checkpoint_metrics':saved['validation_metrics'],
                          'train_metrics':train_m,'validation_metrics':valid_m,'test_metrics':test_m}
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

results=pd.DataFrame(all_rows)
results.to_csv(REPORTS/'three_seed_all_metrics.csv',index=False)
metrics=['NDCG@10','Recall@10','MRR','Accuracy@1','ConceptRecall@10','Recall@20']
requested=results[['Seed','Split','BestEpoch']+metrics]
requested.to_csv(REPORTS/'three_seed_requested_metrics.csv',index=False)
print('\nPer-seed results');print(requested.to_string(index=False))

summary=results.groupby('Split')[metrics].agg(['mean','std']).reset_index()
summary.to_csv(REPORTS/'three_seed_mean_std.csv',index=False)
print('\nThree-seed mean ± standard deviation');print(summary.to_string(index=False))

# Graph 1: learning curves for all three seeds.
fig,axes=plt.subplots(1,2,figsize=(14,5))
for seed,h in histories.items():
    axes[0].plot(h.Epoch,h.TrainLoss,label=f'Seed {seed}')
    axes[1].plot(h.Epoch,h['Val_NDCG@10'],label=f'Seed {seed}')
axes[0].set(title='Training loss by seed',xlabel='Epoch',ylabel='Loss')
axes[1].set(title='Validation NDCG@10 by seed',xlabel='Epoch',ylabel='NDCG@10')
for ax in axes:ax.grid(alpha=.25);ax.legend()
fig.tight_layout();fig.savefig(REPORTS/'01_learning_curves.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 2: split comparison with seed variability.
means=results.groupby('Split')[metrics].mean().reindex(['Train','Validation','Test'])
stds=results.groupby('Split')[metrics].std().reindex(['Train','Validation','Test'])
fig,ax=plt.subplots(figsize=(14,6));means.T.plot.bar(yerr=stds.T,ax=ax,capsize=3)
ax.set(title='Train, validation and test metrics (mean ± SD, 3 seeds)',ylabel='Score',ylim=(0,1.05))
ax.grid(axis='y',alpha=.25);plt.xticks(rotation=20);fig.tight_layout()
fig.savefig(REPORTS/'02_split_metric_comparison.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 3: test stability across seeds.
test_rows=results[results.Split=='Test'].set_index('Seed')[metrics]
fig,ax=plt.subplots(figsize=(13,5));test_rows.plot(marker='o',ax=ax)
ax.axhline(TARGET_TEST_NDCG,color='red',linestyle='--',label='0.90 target')
ax.set(title='Test metric stability across seeds',ylabel='Score',ylim=(0,1.05));ax.grid(alpha=.25);ax.legend(ncol=4)
fig.tight_layout();fig.savefig(REPORTS/'03_test_seed_stability.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 4: HPO trial values.
complete=trials[trials.state=='COMPLETE'] if 'state' in trials else trials
fig,ax=plt.subplots(figsize=(10,5));ax.plot(complete.number,complete.value,marker='o')
ax.axhline(.90,color='red',linestyle='--',label='0.90 target');ax.set(title='Validation NDCG@10 across HPO trials',xlabel='Trial',ylabel='NDCG@10')
ax.grid(alpha=.25);ax.legend();fig.tight_layout();fig.savefig(REPORTS/'04_hpo_trials.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 5: Optuna parameter importance.
try:
    importance=optuna.importance.get_param_importances(study)
    imp=pd.Series(importance).sort_values()
    fig,ax=plt.subplots(figsize=(9,6));imp.plot.barh(ax=ax);ax.set(title='Hyperparameter importance',xlabel='Importance')
    fig.tight_layout();fig.savefig(REPORTS/'05_hyperparameter_importance.png',dpi=200,bbox_inches='tight');plt.close(fig)
except Exception as error:print('Importance graph skipped:',error)

test_ndcg=results[results.Split=='Test']['NDCG@10']
verdict={'target':TARGET_TEST_NDCG,'test_mean':float(test_ndcg.mean()),'test_std':float(test_ndcg.std()),
         'all_seeds_above_target':bool((test_ndcg>=TARGET_TEST_NDCG).all())}
json.dump({'seeds':SEEDS,'best_hyperparameters':study.best_params,'hpo_best_validation_ndcg':study.best_value,
           'runs':manifests,'target_verdict':verdict},open(REPORTS/'three_seed_manifest.json','w'),indent=2)
print('\nTarget verdict:',verdict)
if not verdict['all_seeds_above_target']:
    print('The honest test result is below 0.90 for at least one seed; no metric was altered.')

import shutil
zip_path=shutil.make_archive('/kaggle/working/Three_Seed_Tuned_BCE_SASRec_outputs','zip',root_dir=str(OUT.parent),base_dir=OUT.name)
print('Outputs:',REPORTS);print('ZIP:',zip_path)


Detected processed data: /kaggle/working/processed
Output directory: /kaggle/working/tuned_bce_sasrec
Device: cuda
GPU: Tesla T4
{
  "root": "/kaggle/input",
  "seed": 42,
  "max_len": 50,
  "max_concepts_per_video": 12,
  "hidden_dim": 128,
  "transformer_layers": 2,
  "attention_heads": 4,
  "feedforward_dim": 512,
  "dropout": 0.1,
  "time_buckets": 32,
  "negatives": 50,
  "batch_size": 128,
  "eval_batch_size": 128,
  "max_epochs": 25,
  "minimum_epochs": 10,
  "early_stopping_patience": 5,
  "early_stopping_min_delta": 0.0001,
  "learning_rate": 0.001,
  "weight_decay": 1e-05,
  "concept_loss_weight": 0.2,
  "completion_loss_weight": 0.1,
  "gradient_clip": 5.0,
  "use_mixed_precision": false,
  "ks": [
    5,
    10,
    20
  ],
  "num_workers": 2
}
{'train': 152022, 'validation': 17959, 'test': 17959, 'users': 17959, 'videos': 4720}


,split,interactions,users,videos
0,train,152022,17959,4720
1,validation,17959,17959,2881
2,test,17959,17959,2793


,check,passed
0,train_before_or_at_validation,True
1,validation_before_or_at_test,True
2,validation_items_in_training_catalog,True
3,test_items_in_training_catalog,True
4,same_validation_and_test_users,True


,mean,std
watched_seconds,5.477146,0.598796
playback_seconds,5.279094,0.677260
duration_seconds,5.771293,0.459267
completion_ratio,0.795604,0.233253
segment_count,1.168920,0.493078
engagement_weight,0.771684,0.197891


{'concepts': 14994, 'concept_video_edges': 33302, 'courses': 11, 'videos_with_concepts': 2566, 'videos_with_courses': 253}
Leakage-free evaluation users: 17959
Training prefix examples: 134063 batches per epoch: 1048


...

hpo_trial_03 01/10:   0%|          | 0/1048 [00:00<?, ?it/s]

Validation:   0%|          | 0/141 [00:00<?, ?it/s]

hpo_trial_03 epoch 1: loss=3.4199, val NDCG@10=0.5069, val Recall@10=0.5769


hpo_trial_03 02/10:   0%|          | 0/1048 [00:00<?, ?it/s]

Validation:   0%|          | 0/141 [00:00<?, ?it/s]